# imports

In [44]:
import mysql.connector
import pandas as pd
import numpy as np

from pathlib import Path


In [ ]:
# Path relative to Scripts/
data_dir = Path("../Data")
input_file = data_dir / "2026-06-29_2_bank_dataset_cleaned.csv"

df = pd.read_csv(input_file, encoding="utf-8-sig", index_col=0)

print(f"Dataset loaded from: {input_file.resolve()}")
print(f"Shape: {df.shape}")

Dataset loaded from: C:\Users\nowan\Documents\itacademy\Simulador\ProjecteData\Equip_32\Data\2026-06-29_bank_dataset_cleaned.csv
Shape: (10630, 19)


In [46]:
df = df.reset_index()  # converteix l'índex en columna normal
df.rename(columns={'index': 'id'}, inplace=True)  # per si el nom no és 'id'

In [47]:
df

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit,no_previous_contact,had_previous_contact
0,1,59.0,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,1,1,0
1,2,56.0,admin.,married,secondary,no,45,no,no,unknown,5,may,1391,1,-1,0,unknown,1,1,0
2,3,41.0,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,1,1,0
3,4,55.0,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,1,1,0
4,5,54.0,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10625,10638,33.0,technician,married,secondary,no,218,yes,yes,telephone,2,mar,169,4,-1,0,unknown,0,1,0
10626,10639,42.0,management,single,tertiary,no,1146,yes,no,unknown,15,may,98,2,-1,0,unknown,0,1,0
10627,10640,31.0,unemployed,single,unknown,no,167,no,no,cellular,20,nov,316,1,-1,0,unknown,0,1,0
10628,10641,30.0,blue-collar,single,secondary,yes,447,no,no,cellular,19,nov,426,2,189,6,failure,0,0,1


# Data Transformations

# 1 Demografical clustering

In [48]:
# Age_group
bins   = [17, 25, 35, 50, 65, 100]
labels = [
    "Young (18-25)",
    "Young Adult (26-35)",
    "Adult (36-50)",
    "Middle-Aged (51-65)",
    "Senior (65+)"
]

df["age_group"] = pd.cut(
    df["age"],
    bins=bins,
    labels=labels,
    right=True    # right-closed intervals: (17,25] includes 25
)

In [49]:
counts = df["age_group"].value_counts().sort_index()
print(counts)
print(f"\nUnassigned (NaN): {df['age_group'].isna().sum()}")

age_group
Young (18-25)           437
Young Adult (26-35)    3735
Adult (36-50)          4097
Middle-Aged (51-65)    1969
Senior (65+)            392
Name: count, dtype: int64

Unassigned (NaN): 0


In [50]:
age_summary = (
    df.groupby("age_group", observed=True)["deposit"]
    .value_counts(normalize=True)
    .unstack()
    .rename(columns={1: "pct_yes", 0: "pct_no"})
    .round(4)
)

age_summary["n_clients"] = df.groupby("age_group", observed=True).size()
print(age_summary)

deposit              pct_no  pct_yes  n_clients
age_group                                      
Young (18-25)        0.2700   0.7300        437
Young Adult (26-35)  0.5001   0.4999       3735
Adult (36-50)        0.5641   0.4359       4097
Middle-Aged (51-65)  0.4962   0.5038       1969
Senior (65+)         0.1862   0.8138        392


In [51]:
# Financial Burden

# "unknown" is treated as 0 (no burden assumed)
# This is a deliberate modelling choice — document it in the notebook

burden_map = {"yes": 1, "no": 0, "unknown": 0}

df["default_score"]  = df["default"].map(burden_map)
df["housing_score"]  = df["housing"].map(burden_map)
df["loan_score"]     = df["loan"].map(burden_map)

In [52]:
df["financial_burden"] = (
    df["default_score"] +
    df["housing_score"] +
    df["loan_score"]
)

In [53]:
burden_labels = {
    0: "No burden",
    1: "Low burden",
    2: "Medium burden",
    3: "High burden"
}

df["financial_burden_label"] = df["financial_burden"].map(burden_labels)

In [54]:
burden_summary = (
    df.groupby("financial_burden_label")["deposit"]
    .value_counts(normalize=True)
    .unstack()
    .rename(columns={1: "pct_yes", 0: "pct_no"})
    .round(4)
)

burden_summary["n_clients"] = df.groupby("financial_burden_label").size()

# Sort by score for readability
burden_summary = burden_summary.reindex(burden_labels.values())
print(burden_summary)

deposit                 pct_no  pct_yes  n_clients
financial_burden_label                            
No burden               0.3799   0.6201       5025
Low burden              0.6041   0.3959       4729
Medium burden           0.6628   0.3372        854
High burden             0.6818   0.3182         22


In [55]:
# education

print(df["education"].value_counts())
print(f"\nUnknown count: {(df['education'] == 'unknown').sum()}")


education
secondary    5214
tertiary     3526
primary      1417
unknown       473
Name: count, dtype: int64

Unknown count: 473


In [56]:
# Ordinal scale: unknown → NaN (excluded from ranking)
# primary=1, secondary=2, tertiary=3

education_order = {
    "primary"   : 1,
    "secondary" : 2,
    "tertiary"  : 3,
    "unknown"   : None
}

df["education_rank"] = df["education"].map(education_order)

In [57]:
education_labels = {
    "primary"   : "Primary",
    "secondary" : "Secondary",
    "tertiary"  : "Tertiary",
    "unknown"   : "Unknown"
}

df["education_label"] = df["education"].map(education_labels)

In [58]:
edu_summary = (
    df.groupby("education_label")["deposit"]
    .value_counts(normalize=True)
    .unstack()
    .rename(columns={1: "pct_yes", 0: "pct_no"})
    .round(4)
)

edu_summary["n_clients"] = df.groupby("education_label").size()

# Sort by ordinal rank
order = ["Primary", "Secondary", "Tertiary", "Unknown"]
edu_summary = edu_summary.reindex(order)
print(edu_summary)

deposit          pct_no  pct_yes  n_clients
education_label                            
Primary          0.5829   0.4171       1417
Secondary        0.5307   0.4693       5214
Tertiary         0.4348   0.5652       3526
Unknown          0.4672   0.5328        473


In [59]:
#jobs

print(df["job"].value_counts())
print(f"\nUnknown count: {(df['job'] == 'unknown').sum()}")

job
management       2443
blue-collar      1829
technician       1735
admin.           1278
services          880
retired           754
self-employed     388
student           353
unemployed        338
entrepreneur      307
housemaid         257
unknown            68
Name: count, dtype: int64

Unknown count: 68


In [60]:
job_profile_map = {
    "admin."       : "White Collar",
    "management"   : "White Collar",
    "technician"   : "White Collar",
    "blue-collar"  : "Blue Collar",
    "housemaid"    : "Blue Collar",
    "services"     : "Blue Collar",
    "entrepreneur" : "Self-Employed",
    "self-employed": "Self-Employed",
    "retired"      : "Retired",
    "student"      : "Student",
    "unemployed"   : "Unemployed/Unknown",
    "unknown"      : "Unemployed/Unknown"
}

df["job_profile"] = df["job"].map(job_profile_map)

In [61]:
job_summary = (
    df.groupby("job_profile")["deposit"]
    .value_counts(normalize=True)
    .unstack()
    .rename(columns={1: "pct_yes", 0: "pct_no"})
    .round(4)
)

job_summary["n_clients"] = df.groupby("job_profile").size()

order = ["White Collar", "Blue Collar", "Self-Employed", "Unemployed", "Retired", "Student", "Unknown"]
job_summary = job_summary.reindex(order)
print(job_summary)

deposit        pct_no  pct_yes  n_clients
job_profile                              
White Collar   0.4927   0.5073     5456.0
Blue Collar    0.6001   0.3999     2966.0
Self-Employed  0.5540   0.4460      695.0
Unemployed        NaN      NaN        NaN
Retired        0.3170   0.6830      754.0
Student        0.2408   0.7592      353.0
Unknown           NaN      NaN        NaN


In [62]:
month_map = {
    'jan': 'January', 'feb': 'February', 'mar': 'March',
    'apr': 'April', 'may': 'May', 'jun': 'June',
    'jul': 'July', 'aug': 'August', 'sep': 'September',
    'oct': 'October', 'nov': 'November', 'dec': 'December'
}
df['month'] = df['month'].map(month_map)

In [63]:
df['week_of_month'] = ((df['day'] - 1) // 7) + 1
# Resultat: 1, 2, 3, 4, 5

In [64]:
bins = [0, 3, 10, 20, 30, float('inf')]
labels = ['0-3', '4-10', '11-20', '21-30', '+30']
df['campaign_group'] = pd.cut(df['campaign'], bins=bins, labels=labels)

In [65]:
# Opció 2: Reset de l'índex com a columna neta
df = df.reset_index(drop=True)
df.index = df.index + 1  # comença des de 1 en lloc de 0



# CSV export

In [ ]:
# Drop auto-generated index column if it was imported as a column
if "index" in df.columns:
    df = df.drop(columns=["index"])

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])
 

# Drop intermediate scoring columns
cols_to_drop = [
    "default_score",
    "housing_score", 
    "loan_score"
]

df_export = df.drop(columns=cols_to_drop)

# Export transformed dataset
output_dir = Path("../Data")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "2026-06-29_3_bank_dataset_transformed.csv"

# Export
df_export.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"CSV saved to: {output_file.resolve()}")
print(f"Shape: {df_export.shape}")
print(f"Columns: {df_export.columns.tolist()}")

CSV saved to: C:\Users\nowan\Documents\itacademy\Simulador\ProjecteData\Equip_32\Data\2026-06-29_bank_dataset_transformed.csv
Shape: (10630, 28)
Columns: ['id', 'age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'deposit', 'no_previous_contact', 'had_previous_contact', 'age_group', 'financial_burden', 'financial_burden_label', 'education_rank', 'education_label', 'job_profile', 'week_of_month', 'campaign_group']
